<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP11A/MP11A_faster_a_to_b.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP11A · Faster from A to B

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex11-vision-navigation/")
PROJECT = (['mp11a.py', 'r2twin.py', 'MP11A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP11A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex11-vision-navigation", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex11-vision-navigation", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem', 'mp11a', 'r2twin']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem as hs
import mp11a as mp
torch.set_default_dtype(torch.float32)                   # no second derivative here: single precision, as in Ex11
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP11A_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex11's problem.py, by its name in Ex11
import r2twin as rt                                      # the ROSMASTER R2 twin (r2twin.py, fetched above)

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

A mobile robot that drives among obstacles - a warehouse vehicle, a delivery robot, a car on a test track - answers two questions many times a second: **is something in my way, and where exactly is it?** The first decides whether to swerve; the second decides how far. Ex11 answered the first with a network and took the second from the course: the race placed every obstacle for both cars alike. A real car must measure both, and the better it measures *where*, the closer and the faster it can pass.

This mini project asks for both answers **from the colour camera alone**, on a larger car and a harder course than Ex11's: the **Yahboom ROSMASTER R2**, driven one lap round a closed track with bends, a chicane and up to eight obstacles of four kinds, in a room whose light, shadows and sun change in five seeded levels. The network you build reads one camera frame and returns *blocked or not* and *where the obstacle stands*; a planner turns that into steering and speed. The goal is Ex11's mini project topic, **faster from A to B**: a lap at 1.5 m/s with no collision, close to the lap a car with perfect perception drives.

Everything runs in Colab on a **digital twin** of the R2: the frames are rendered from where the car stands, so the true answer to every question is known. No real car and no ROS are needed (the lecturer, 27 October 2026).

## A2 · The car: the ROSMASTER R2 and its twin

The R2 is a four-wheel robot car with **Ackermann steering** (the front wheels turn, as on a road car), a Jetson Orin Nano Super, a **SLAM A1 LiDAR** on its top deck and an **Astra Pro Plus depth camera** at the front, which carries a colour camera beside its depth sensor. This project uses the **colour camera only**; MP11B fuses all three sensors.

The twin is the file `r2twin.py`, fetched beside this notebook. It extends Ex11's `problem.py`: the same kinematic bicycle (the car turns about a point on the line of its rear axle), the same speed lag, Ex11's colours for the floor, the tape and the obstacles. **What the vendor states is used as stated; everything else is marked ASSUMED** and would be measured on the car before the twin is trusted (course rule: do not invent hardware figures). The cell prints the table and draws the twin in 3-D on the track, its LiDAR's scan plane in red.

In [ ]:
# The ROSMASTER R2 twin: what the vendor states and what is assumed
for what, value, source in rt.spec():
    print(f"{what:>19}  {value:<72} {source}")
tr = rt.Track(1000, 1)                                   # a test track at noise level 1
s = tr.obstacle_s[1][0] - 1.3                            # 1.3 m before its second obstacle
x, y = tr.point(s, 0.1)
ax = rt.view3d(tr, x, y, tr.heading(s), title="the ROSMASTER R2 twin")
plt.show()

**What you should see.** Nineteen rows: the controller, the Ackermann steering, the largest speed of 1.8 m/s, the 65 mm wheels, the camera's field of view of 66.1 × 40.2 degrees and its 1920 × 1080 stream, the depth range of 0.6 to 8 m, the LiDAR's 360 ranges a turn are the vendors'; the wheelbase (0.23 m), the width (0.24 m), the steering limit (25 degrees), the camera's mount (17 cm high, 10 degrees down) and every noise are ASSUMED. Then the R2 drawn in 3-D on the track, seen from behind: the aluminium chassis, the top deck with the Jetson and the LiDAR, the 7-inch screen, the depth camera at the front, and the LiDAR's beams fanning out at 20 cm above the floor. The drawing follows the vendor's photograph; its shapes are not measurements.

## A3 · The track and its obstacles

A closed track, 1.1 m wide between two taped edges, about 21 m round: a start straight, a **chicane** (a left-right jog of 0.55 to 0.75 m), a long right-hand bend, an **S-bend** and a long left-hand bend back to the start. A and B are the same line: one lap. Every track is drawn from a **seed**: the shape moves by up to 12 cm, and 4 to 8 obstacles stand at least 1.6 m apart along it, up to 25 cm from the centre line, each of radius 7 to 10 cm and of one of four kinds:

| kind | looks like | height | what is hard about it |
|---|---|---|---|
| `box` | dark grey | 30 cm | nothing: every sensor sees it |
| `grey` | the floor's own colour | 30 cm | the camera barely sees it against the floor |
| `low` | dark brown | 10 cm | under the LiDAR's plane (MP11B); small in the frame |
| `black` | glossy black | 30 cm | returns little light to the LiDAR and the depth camera (MP11B) |

The car localises itself on the track by its **odometry and IMU** (both vendor-stated), assumed exact in the twin: it knows the track's shape and where it is on it, but not where the obstacles are. That is what the camera is for.

In [ ]:
# One test track from above: the taped edges, the start line A = B, the obstacles, shadows and sun
tr = rt.Track(1000, 2)
tr.describe()
fig, ax = plt.subplots(figsize=(10, 5.5))
rt.draw_track(tr, ax)
ax.set_title("test track 1000 at noise level 2: grey circles are shadows, yellow ones sun on the floor", fontsize=9)
plt.show()

**What you should see.** The track from above with its taped edges, the start line in blue, and its obstacles: test track 1000 has a lap of 20.91 m, its tightest bend a radius of 1.01 m, and four obstacles: three dark boxes and a low block. The light red ring round each obstacle is the car's half-width: the car's centre must stay outside it. At noise level 2 the room is a little dim (light 0.84 of full), three shadows lie on the floor and two patches of sun.

## A4 · The noise, at five levels

A real room is never the room the network was trained in. The twin makes it worse in five seeded levels, each a step that an engineer would meet on some day in a real lab (all **ASSUMED**, chosen to make the problem harder step by step):

| level | the room | pixel noise | shadows | sun patches | motion blur |
|---|---|---|---|---|---|
| 0 | a clean lab, full light | 1 % | none | none | none |
| 1 | a normal lab, light 0.85 - 1 | 2 % | 2 | 1 | 0.6 rows per m/s |
| 2 | a difficult day, light 0.70 - 0.85 | 3.5 % | 3 | 2 | 1.2 rows per m/s |
| 3 | a dim room with sun, light 0.50 - 0.65 | 5 % | 4 | 3 | 1.8 rows per m/s |
| 4 | extreme, light 0.35 - 0.50 | 7 % | 5 | 4 | 2.4 rows per m/s |

Pixel noise grows in a dimmer room (a darker image is amplified more), the light also changes across the frame, and **motion blur** smears the frame down the image at speed (the floor moves towards the car while the shutter is open). A **shadow is not an obstacle**, but it is dark like one; **sun on the floor** is bright. You train at levels 0 to 2; level 3 is **never seen** in training and tells you how your network fails when the room changes.

In [ ]:
# The same place on test track 1000 at the five noise levels, the car at 1.5 m/s
s0 = tr.obstacle_s[0][0] - 1.0                          # 1 m before the first obstacle
fig, ax = plt.subplots(1, 5, figsize=(16, 2.2))
for lv in range(5):
    t = rt.Track(1000, lv)
    x, y = t.point(s0, 0.05)
    ax[lv].imshow(rt.render_rgb(t, x, y, t.heading(s0), np.random.default_rng(lv), speed=1.5))
    ax[lv].set_title(f"level {lv}: light {t.light:.2f}", fontsize=9); ax[lv].axis("off")
plt.show()

**What you should see.** One frame, 112 × 64 pixels, five times: the obstacle ahead, the taped edges curving away, the room's wall above the horizon. From level 0 to level 4 the frame darkens (light 1.00, 0.96, 0.84, 0.53, 0.47 on this track), grows grainier and blurs; shadows and sun patches appear on the floor where they fall in view.

## A5 · What the car must perceive, and how it drives

**Blocked** means: an obstacle's centre stands between the car's front axle and 1.5 m beyond it (`rt.RANGE`), within 0.55 m plus its radius to either side of the car's axis (the zone `rt.LANE`, the track's half-width, because in a bend an obstacle on the track ahead stands well to the side of the car's axis), and inside the camera's field of view. **Where** is that obstacle's position, the nearest such one: its distance **forward** from the rear axle and **sideways** (left positive) in metres, in the car's frame. Both are computed from the twin's geometry by `rt.truth(...)`, exactly.

**The drive**, given to you (`rt.drive`), decides ten times a second, as Ex11 does, and a decision takes one period, so the wheels follow the command from the decision before. Each decision:

1. reads the latest colour frame and asks the perception for *(blocked, forward, sideways)*;
2. the **planner** turns that into a speed and a sideways target on the track. Two planners are given, both Ex11's ideas on the track:
   * `rt.SteerAround`: Ex11's rule - when blocked, aim 0.40 m beside the obstacle, on the side the car is already on, at 60 % speed, until the car's rear is past it;
   * `rt.FieldAround`: Ex11's potential field - every obstacle reported twice within a second is remembered at the mean of its sightings, and the car aims at the lowest point across the track of $U(d) = d^2 + \sum_i K/g_i(d) + K_w/g_\mathrm{edge}(d)$, $g_i$ the sideways gap between the car's side and obstacle $i$ with a 10 cm margin, at full speed; it slows to half when an obstacle within 1 m is not yet cleared;
3. **pure pursuit** steers towards that target, one look-ahead distance along the track;
4. in a bend the speed is held so that the sideways acceleration stays under 1 m/s² (`rt.bend_speed`): there the camera sees less of the track ahead.

A car that touches an obstacle stops the run (a **collision**); one whose centre leaves the track by more than 0.1 m has **left the track**. The lap time is from A back to A.

## A6 · Why there is no exact answer, and what the ground truth is

Ex11's exercises could be checked against a formula; a camera cannot. What a frame shows depends on the shape of the track, the obstacles' places, the light, the shadows and the noise, and no closed form says which pixels an obstacle covers or how a lap unfolds when each decision depends on the frame before. So the ground truth is built in two parts, both **numerical and exact to what they claim**:

* **the labels**: blocked and where for every frame, from the twin's geometry (`rt.truth`), exact to rounding;
* **the lap with perfect perception**: the same planner fed those exact labels, driven round every test track; its lap time and smallest gap are what the best possible perception would give. The car is moved by a time-stepping (explicit Euler) integration of the bicycle model, so its convergence in the time step is shown in part C, with Richardson's extrapolation, as course policy C10 asks.

## A7 · What is asked

| quantity | where | required |
|---|---|---|
| **collisions** | the 10 test tracks (seeds 1000 to 1009), never trained on, at noise levels 0, 1 and 2, the potential field at 1.5 m/s: 30 laps | **none** |
| **lap time** | the same 30 laps | the mean within **5 %** of perfect perception's with the same planner and speed (18.13 s, part C) |
| **blocked or not** | 900 held-out frames at levels 0 to 2 (20 tracks never trained on) | for **every kind**, at most **5 %** of obstacles missed; at most **5 %** false stops on free frames |
| **where** | the same frames, where both say blocked | the mean position error at most **10 cm** for every kind |
| **decision time** | one frame, on the Colab T4 | at most **10 ms**, a tenth of the 100 ms between decisions |

Reported, not required: noise level 3, which you do not train on; the conventional camera rule's results against yours; the decision time of the network against the conventional rule's on the same runtime.

**Why these numbers.** Zero collisions is the only acceptable safety result. The 5 % on the lap time leaves room for a few false stops (a phantom obstacle makes the car swerve and slow) but not for a car that crawls. Missing 5 % of the frames that show an obstacle is survivable because the car sees each obstacle in ten to fifteen frames on its way; the position error of 10 cm is half the 10 cm margin plus the 10 cm the planner assumes for an obstacle's radius beyond the real 7 to 10 cm. The conventional rule meets none of these (part C).

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

$$\text{frame}\ (3 \times 64 \times 112,\ \text{0 - 255})\ \longmapsto\ (\ell,\ f,\ s)$$

* the input is one colour frame as the camera gives it, red-green-blue, 64 rows by 112 columns, integers from 0 to 255;
* $\ell$ is the **logit** of the probability that the path is blocked, $p = 1/(1 + e^{-\ell})$; the car counts the path as blocked when $p > 0.5$;
* $f$ and $s$ are **where** the nearest blocking obstacle stands, in metres from the rear axle: forward, from about 0.3 to 1.8, and sideways, left positive, from about -0.9 to 0.9. They mean something only when the path is blocked.

One network gives all three, as Ex11's network gave *blocked* and the position head adds *where*: the convolutional layers find the obstacle once, two small outputs read it.

## B2 · What the network is trained with

Every frame comes with its exact labels, so the training is supervised: two loss terms, one per question.

| loss term | on which frames | what it asks |
|---|---|---|
| $\mathcal{L}_\mathrm{blocked} = -\overline{y \ln p + (1 - y)\ln(1 - p)}$ | every frame | the cross-entropy of L6.1 and Ex11: $p$ close to the label $y$ (1 blocked, 0 free) |
| $\mathcal{L}_\mathrm{where} = \overline{H\big((\hat f - f)/0.5\big) + H\big((\hat s - s)/0.5\big)}$ | the blocked frames only | the position, scaled by 0.5 m to order one, with the Huber loss $H$ (quadratic below 0.1, linear above) |

The position loss is **masked**: a free frame has no obstacle, so it has no position to learn, and asking for one would teach the network nonsense. The Huber loss keeps a few badly placed frames (an obstacle half out of view) from dominating the mean.

**What can be built in.** The camera looks at a flat floor from a known height and tilt, so each row of the frame below the horizon sees the floor at a known distance: the bottom row 0.29 m ahead of the camera, the horizon at infinity. An obstacle stands on the floor, so the row of its foot says how far it is, and its column says how far to the side. A convolutional network learns this mapping from the data; the cell below draws it, because it is what your network's *where* has to rediscover. You may also feed the network the row distances as an extra input, or normalise the frame's brightness against the room's light - your choice in part D.

In [ ]:
# The flat floor seen by the camera: the distance each row of the frame sees, and the floor to the side
rows = np.arange(rt.CAM_H)
ahead = np.array([np.nanmean(r) if np.isfinite(r).any() else np.nan for r in rt.FLOOR_F]) - rt.CAM_AHEAD   # m ahead of the camera, per row (nan above the horizon)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].plot(rows, ahead, "o-", ms=3)
ax[0].set_xlabel("row of the frame (0 top, 63 bottom)"); ax[0].set_ylabel("floor seen, ahead of the camera [m]")
ax[0].set_ylim(0, 5); ax[0].axhline(rt.RANGE + rt.WHEELBASE - rt.CAM_AHEAD, color="r", lw=0.8)
ax[0].text(20, 1.25, "the end of the blocked zone", color="r", fontsize=9)
im = ax[1].imshow(np.where(rt.LANE_MASK, rt.FLOOR_F, np.nan), cmap="viridis")
plt.colorbar(im, ax=ax[1], label="floor ahead of the rear axle [m]"); ax[1].set_title("the zone watched, on the frame", fontsize=9)
plt.tight_layout(); plt.show()
print(f"horizon at row {rt.HORIZON:.1f}; the bottom row sees the floor {rt.NEAREST_FLOOR:.2f} m ahead of the camera; "
      f"the zone ends {rt.RANGE + rt.WHEELBASE - rt.CAM_AHEAD:.2f} m ahead of it")

**What you should see.** Left, the distance per row: from the bottom row's 0.29 m it grows slowly up the frame and then steeply towards the horizon at row 16.3; the far end of the blocked zone (1.41 m ahead of the camera) lies only about ten rows below the horizon, so a distant obstacle's foot is a few pixels high and its position is hard to read. Right, the zone watched by the conventional rule (part C) on the frame: the floor of the track's width ahead, coloured by its distance.

## B3 · The training data

`mp.training_data()` renders the frames from the twin: **120 tracks (seeds 0 to 119), noise levels 0, 1 and 2, 40 poses on each - 14 400 frames**. A pose is on the track, up to 0.35 m off its centre line, turned by up to 17 degrees, at a speed from 0 to 1.8 m/s (for the motion blur); on 65 % of the poses an obstacle stands 0.2 to 1.8 m ahead, so that about half the frames are blocked. Each frame comes with *blocked*, *where*, the obstacle's *kind*, whether a shadow lies in the zone with no obstacle, the level and the track. Every pose and every pixel of noise is drawn from a seed, so the data are the same on every run.

In [ ]:
# The training frames, rendered from the twin
t0 = time.perf_counter()
DATA = mp.training_data()
print(f"{len(DATA['blocked'])} frames in {time.perf_counter() - t0:.0f} s; blocked {100 * DATA['blocked'].mean():.1f} %; "
      + ", ".join(f"{k} {int(np.sum(DATA['kind'] == k))}" for k in rt.KIND_NAMES)
      + f"; shadow-only {int(DATA['shadow'].sum())}")
fig, ax = plt.subplots(2, 6, figsize=(16, 3.6))
for a, k in zip(ax.ravel(), np.random.default_rng(1).choice(len(DATA["blocked"]), 12, replace=False)):
    a.imshow(DATA["rgb"][k].transpose(1, 2, 0)); a.axis("off")
    lab = f"{DATA['kind'][k]} at {DATA['where'][k][0]:.2f}, {DATA['where'][k][1]:+.2f} m" if DATA["blocked"][k] else "free"
    a.set_title(f"level {DATA['level'][k]}: {lab}", fontsize=7)
plt.tight_layout(); plt.show()

**What you should see.** 14 400 frames rendered in 263 s on one CPU thread shared with other jobs; 48.0 % blocked: 2 119 dark boxes, 1 478 grey boxes, 1 637 low blocks and 1 681 black ones, and 689 frames with only a shadow in the zone. Twelve examples with their labels: the obstacle's kind and where it stands, forward and sideways from the rear axle.

## B4 · The test data

Two kinds of test, both on tracks the network never sees:

* `mp.test_samples()`: **20 tracks (seeds 900 to 919), 15 poses each at noise levels 0, 1, 2 and 3: 1 200 frames**, scored frame by frame. Their labels are also in the ground-truth file `MP11A_truth.npz`, and the cell checks that the frames rendered here carry them;
* `mp.TEST_TRACKS`: **10 tracks (seeds 1000 to 1009)**, driven in closed loop, one lap each at every level.

In [ ]:
# The held-out frames, and the check against the ground-truth file
G = np.load("MP11A_truth.npz")
TEST = mp.test_samples()
same = np.array_equal(TEST["blocked"], G["sample_blocked"]) and np.allclose(np.nan_to_num(TEST["where"]), np.nan_to_num(G["sample_where"]), atol=1e-5)
print(f"{len(TEST['blocked'])} held-out frames; labels as in the truth file: {same}")
for lv in mp.TEST_LEVELS:
    m = TEST["level"] == lv
    print(f"  level {lv}: {m.sum()} frames, blocked {int(TEST['blocked'][m].sum())}: "
          + ", ".join(f"{k} {int(np.sum(TEST['kind'][m] == k))}" for k in rt.KIND_NAMES) + f"; shadow-only {int(TEST['shadow'][m].sum())}")

**What you should see.** 1 200 frames, labels as in the truth file: `True`. At each level 300 frames, 147 to 164 blocked, of every kind, and at levels 1 to 3 some frames with only a shadow in the zone - the frames on which a rule that looks for dark pixels stops for nothing.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two references, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the conventional vision an engineer would write: Ex11's dark-pixel rule on the R2's frame, the obstacle's foot read off the flat floor | the twin's geometry: exact labels, and the lap driven with perfect perception |
| its job | to be **compared** with your network - accuracy per kind, lap, collisions - and **timed** per decision | to **score** both |
| planner | Ex11's steer-around rule at 1.0 m/s (Ex11's speed), and the potential field at 1.5 m/s | the potential field at 1.5 m/s, and the steer-around rule at 1.0 m/s |

## C2 · The conventional rule, step by step

`rt.camera_rule(frame)` is Ex11's dark-pixel rule, given a position:

1. **Grey.** The frame is turned into brightness, the mean of red, green and blue, from 0 to 1.
2. **The floor's brightness.** The median over every pixel that sees the floor: most of the floor is bare, so the median is the bare floor's brightness in this frame, whatever the room's light.
3. **Dark pixels in the zone.** In the zone watched (the floor of the track's width ahead, up to 0.3 m beyond the blocked zone), a pixel darker than the floor by 0.15 counts as dark.
4. **Blocked** when more than 0.4 % of the zone's pixels are dark.
5. **Where.** The dark pixel lowest in the frame is the obstacle's foot, nearest the car; the flat floor gives its distance forward and sideways, and the obstacle's centre is taken 8.5 cm (a typical radius) further.

The two numbers in steps 3 and 4 were chosen once, on frames like these. The rule's weaknesses are those of Ex11's: a **grey** obstacle is as light as the floor, so it has no dark pixels; a **shadow** is dark, so it is taken for an obstacle; in a **dim room** a dark box differs from the dark floor by less than the threshold. The cell shows it at work on three frames.

In [ ]:
# The conventional rule on three held-out frames: a dark box, a grey box, a shadow with no obstacle
picks = [int(np.where((TEST["kind"] == "box") & (TEST["level"] == 1))[0][0]),
         int(np.where((TEST["kind"] == "grey") & (TEST["level"] == 1))[0][0]),
         int(np.where(TEST["shadow"])[0][0])]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
for a, k in zip(ax, picks):
    fr = TEST["rgb"][k].transpose(1, 2, 0)
    g = fr.mean(axis=-1) / 255
    dark = rt.LANE_MASK & (g < np.median(g[rt.FLOOR_F == rt.FLOOR_F]) - 0.15)
    a.imshow(fr); a.contour(rt.LANE_MASK, [0.5], colors="cyan", linewidths=0.8)
    a.imshow(np.ma.masked_where(~dark, dark), cmap="autumn", alpha=0.8)
    b, f, s = rt.camera_rule(fr)
    lab = f"{TEST['kind'][k] or 'free'}" + (f" at {TEST['where'][k][0]:.2f}, {TEST['where'][k][1]:+.2f} m" if TEST["blocked"][k] else "")
    a.set_title(f"truth: {lab}\nrule: " + (f"blocked at {f:.2f}, {s:+.2f} m" if b else "free"), fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

**What you should see.** Cyan, the zone watched; red, the pixels the rule calls dark. The dark box is found and placed within a few centimetres; the grey box has no dark pixels, so the rule says *free* with an obstacle ahead; the shadow's pixels are dark, so the rule says *blocked* where nothing stands.

## C3 · The ground truth: converged, and checked

The lap with perfect perception is integrated in time steps. The truth file holds the study: the 10 test tracks driven with the potential field at 1.5 m/s and the time step halved three times, from 4 to 0.5 ms.

In [ ]:
# The convergence of the perfect-perception lap in the time step, from the truth file
print("time step   mean lap over the 10 test tracks   smallest gap")
for dt, t, g in zip(G["conv_dt"], G["conv_time"], G["conv_gap"]):
    print(f"  {1e3 * dt:4.1f} ms     {t:.4f} s                          {100 * g:.2f} cm")
d = np.abs(np.diff(G["conv_time"]))
print("change between steps:", " ".join(f"{v:.1e}" for v in d), "s; ratios", np.round(d[:-1] / d[1:], 1))
print(f"no regular order, so no Richardson extrapolation: the finest step gives {float(G['conv_richardson']):.4f} s; the 2 ms used is "
      f"{abs(G['conv_time'][1] - float(G['conv_richardson'])) * 1e3:.1f} ms from it")
print(f"perfect perception, potential field at {float(G['speed'])} m/s: mean lap {G['field_time'].mean():.2f} s, "
      f"smallest gap {100 * G['field_gap'].min():.1f} cm, no collision: {bool(np.all(np.isfinite(G['field_time'])))}")
print(f"perfect perception, steer-around at {float(G['ref_speed'])} m/s: mean lap {G['steer_time'].mean():.2f} s, "
      f"smallest gap {100 * G['steer_gap'].min():.1f} cm")

**What you should see.** Mean laps of 18.1345, 18.1254, 18.1243 and 18.1229 s at time steps of 4, 2, 1 and 0.5 ms; the changes, 9.1e-03, 1.0e-03 and 1.4e-03 s, do not fall at a regular rate (ratios 8.9 and 0.7), because the car decides only every 0.1 s and a slightly different path can change a decision: the lap is not a smooth function of the step, and Richardson's extrapolation does not apply. What matters is the size: the 2 ms step used is 2.4 ms (0.01 %) from the finest, a thousand times below the 5 % asked. With perfect perception and the potential field at 1.5 m/s the mean lap is **18.13 s**, the smallest gap 9.7 cm, no collision; with the steer-around rule at 1.0 m/s, 28.54 s.

In [ ]:
# The physics checks of the twin, run here
checks = mp.check_sensors()

**What you should see.** Four checks against values computed by hand. With the JetRacer's numbers the R2's motion is Ex11's car exactly (0 m apart: the same equations, the limiting case that reproduces the exercise). With the noise off, the LiDAR's range to an obstacle 2 m ahead matches the exact distance to 2e-08 m and the depth camera's reading of the bare floor about 1 m ahead matches $h/\sin\alpha$ along the camera's axis to 5e-08 m (single-precision rounding). On an empty track at 1 m/s the lap takes 21.20 s against $L/v$ = 21.08 s: pure pursuit does not drive exactly on the centre line, 0.6 % longer.

## C4 · The reference model on the held-out frames

The rule is scored on the 1 200 held-out frames, level by level, with the same measures your network will be scored with, and timed: one decision on one frame, on this runtime's CPU (the rule is a few array operations; it would not run faster on a GPU).

In [ ]:
# The conventional rule on the held-out frames, level by level, and its time per decision
ref_ans = mp.reference_answers(TEST)
REF = {"rates": {}}
print(f"{'level':>5} " + " ".join(f"{'missed ' + k:>13}" for k in rt.KIND_NAMES) + f"{'false stops':>12}{'shadow stops':>13}"
      + " ".join(f"{'where ' + k:>11}" for k in rt.KIND_NAMES))
for lv in mp.TEST_LEVELS:
    m = TEST["level"] == lv
    r = rt.rates(ref_ans[m, 0].astype(int), {k: v[m] for k, v in TEST.items()}, ref_ans[m, 1:])
    REF["rates"][lv] = r
    print(f"{lv:>5} " + " ".join(f"{100 * r['missed ' + k]:12.1f}%" for k in rt.KIND_NAMES)
          + f"{100 * r['false stops']:11.1f}%{100 * r['false stops, shadow']:12.1f}%"
          + " ".join(f"{r['where ' + k + ' [cm]']:8.1f} cm" for k in rt.KIND_NAMES))
frame = TEST["rgb"][0].transpose(1, 2, 0)
REF["t_rule_s"] = middle(lambda: rt.camera_rule(frame))
print(f"one decision of the conventional rule: {REF['t_rule_s'] * 1e3:.3f} ms")

**What you should see.** The rule misses **86.7 to 100 % of the grey boxes** at every level: they have no dark pixels. It finds the dark boxes, the low blocks and most black ones (0 to 13.6 % missed), places them within 3 to 9 cm at levels 0 to 2 - and 7 to 26 cm at level 3, where the dim room shrinks the contrast. It stops for 5 to 11 % of the frames with only a shadow ahead. One decision takes 0.152 ms on this runtime's CPU: a few array operations.

## C5 · The reference model in closed loop

Each of the 10 test tracks is driven once at levels 0, 1 and 2 with the conventional rule: with Ex11's steer-around rule at 1.0 m/s, and with the potential field at 1.5 m/s, the speed asked of your network. About two minutes.

In [ ]:
# The conventional rule driving the test tracks, against perfect perception
t0 = time.perf_counter()
print("conventional rule, steer-around at 1.0 m/s:")
REF["steer"] = mp.closed_loop(lambda tr: rt.conventional(("camera",)), rt.SteerAround, mp.REF_SPEED, verbose=True)
print("conventional rule, potential field at 1.5 m/s:")
REF["field"] = mp.closed_loop(lambda tr: rt.conventional(("camera",)), rt.FieldAround, mp.SPEED, verbose=True)
print(f"perfect perception, potential field at 1.5 m/s: mean lap {G['field_time'].mean():.2f} s, no collision "
      f"({time.perf_counter() - t0:.0f} s for the 60 laps)")
REF["perfect_time"] = float(G["field_time"].mean())

**What you should see.** The conventional rule collides on **7, 9 and 7 of the 10 tracks** at levels 0, 1 and 2 with the steer-around rule at 1.0 m/s, and on 7, 8 and 6 with the field at 1.5 m/s - almost always on a grey box it never saw (the mean laps printed are over the few tracks it finished). Perfect perception with the field at 1.5 m/s finishes all ten, in 18.13 s on average. The 60 laps took 118 s here.

## C6 · A lap with perfect perception, filmed

The R2 in 3-D with its LiDAR's beams, its colour frame and its depth image, and the track from above, through the chicane of test track 1000 at noise level 2, driven with perfect perception and the potential field at 1.5 m/s. About a minute to draw.

In [ ]:
# The film: a few seconds of a lap with perfect perception
from IPython.display import HTML, display
tr = rt.Track(1000, 2)
run = rt.drive(tr, rt.oracle(tr), rt.FieldAround(), mp.SPEED, need=(), record=True)
anim = rt.film(run, tr, title="perfect perception, potential field, 1.5 m/s", start=1.0, seconds=5.0, every=5)
display(HTML(anim.to_html5_video()))                    # needs ffmpeg, which Colab has

**What you should see.** The R2 approaching the chicane and its obstacles, the red LiDAR beams sweeping the room at 20 cm, the colour frame and the depth image changing as it swerves, and its path drawn in orange on the track from above.

---

# Part D · The Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else is given.

**The contract.** Your network is a PyTorch module called `net`, used as `net(frames)`:

| argument | meaning | shape |
|---|---|---|
| `frames` | colour frames as the camera gives them, uint8, 0 - 255, on `DEVICE` | `(n, 3, 64, 112)` |
| returns | the logit of *blocked*, then *forward* and *sideways* in metres | `(n, 3)` |

Any scaling of the input (to floats, to order one, against the room's light) happens inside `net`: the drive and the evaluation hand it raw frames.

## D2 · The loss and a quick score, given

`loss_fn(out, blocked, where)` is section B2's two terms: the cross-entropy on every frame and the masked Huber loss of the position on the blocked frames, the position scaled by 0.5 m. `quick_score()` scores `net` on the held-out frames of levels 0 to 2 in a second: missed per kind, false stops and the mean position error per kind.

In [ ]:
# The loss and the quick score, given ------------------------------------------------
import torch.nn.functional as F_

def loss_fn(out, blocked, where):
    """Cross-entropy on blocked (every frame) + Huber loss of the position (blocked frames only), scaled by 0.5 m."""
    l_b = F_.binary_cross_entropy_with_logits(out[:, 0], blocked)                       # L_blocked: is the path blocked?
    m = blocked > 0.5                                                                  # the position exists only where blocked
    l_w = F_.smooth_l1_loss(out[m, 1:] / 0.5, where[m] / 0.5, beta=0.1) if m.any() else out.sum() * 0   # L_where, Huber
    return l_b + l_w, l_b.detach(), l_w.detach()

def predict(net, frames, batch=256):
    """The network's answers on uint8 frames (n, 3, 64, 112): probability blocked, forward, sideways (NumPy)."""
    out = []
    with torch.no_grad():
        for i in range(0, len(frames), batch):
            out.append(net(torch.as_tensor(frames[i:i + batch], device=DEVICE)).float().cpu().numpy())   # one batch through the network
    o = np.concatenate(out)
    return 1 / (1 + np.exp(-o[:, 0])), o[:, 1:]

def quick_score(threshold=mp.THRESHOLD):
    """Missed per kind, false stops and position error per kind [cm] on the held-out frames of levels 0 - 2."""
    m = np.isin(TEST["level"], mp.REQUIRED_LEVELS)
    p, w = predict(net, TEST["rgb"][m])
    return rt.rates((p > threshold).astype(int), {k: v[m] for k, v in TEST.items()}, w)

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees**: the frame scaled to order one; perhaps normalised against the frame's own floor brightness (the room's light varies by a factor of two between levels 0 and 2), perhaps with the row distance of section B2 as a fourth input channel;
* **its layers**: Ex11's convolutional network is a start - convolutions with stride two, then a dense layer - with **two heads** or one output of three: the logit, and the position in metres;
* **its size**: the frame is small (64 × 112), the network must decide in a few milliseconds on the car.

What parts A to C tell you: the grey box differs from the floor in shading and in its silhouette against the wall, not in brightness; a shadow is dark but flat; the foot's row gives the distance.

In [ ]:
# TODO 1 - your network. net(frames uint8 (n, 3, 64, 112)) -> (n, 3): logit, forward [m], sideways [m]
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, frames):
        ...                                              # return (n, 3)

torch.manual_seed(88)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` on `DATA` (section B3) with `loss_fn`. Decide the optimiser, the learning rate and its schedule, the batch size, the number of passes over the data, whether to augment the frames. Keep the history of the two loss terms for TODO 3, and **measure the training time**: set `TRAIN_SECONDS`, which part E reports as the offline cost. `quick_score()` tells you, in a second, where you stand against the requirements of part A.

The tensors: `DATA["rgb"]` (uint8 frames), `DATA["blocked"]` (0 / 1), `DATA["where"]` (metres, nan where free: replace the nan by 0, the mask ignores them).

In [ ]:
# TODO 2 - your training
history = {"blocked": [], "where": []}
t0 = time.perf_counter()
...                                                      # your optimiser and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s;", {k: round(v, 3) for k, v in quick_score().items()})

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the two loss terms during training, and a few held-out frames - at least one grey box and one shadow - with the network's answer beside the truth. Save each figure you want in the report as `MP11A_report_<name>.png` (in `cc.output_path`): part E embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* its answers on all 1 200 held-out frames (levels 0 to 3): the probability of *blocked* and the position;
* **one lap of each of the 10 test tracks at levels 0, 1, 2 and 3**, the network deciding with the potential field at 1.5 m/s: the lap times, the smallest gaps, the collisions (about two minutes);
* the **decision time** on this runtime: one frame through the network on `DEVICE`, the middle of five timings after a warm-up, against the conventional rule's (part C).

Set `TRAIN_SECONDS` to the training time you measured; it is reported beside the online times.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
p_test, w_test = predict(net, TEST["rgb"])                       # the held-out frames, levels 0 - 3
t0 = time.perf_counter()
loop = mp.closed_loop(lambda tr: mp.network_perceive(net, DEVICE), rt.FieldAround, mp.SPEED, levels=mp.TEST_LEVELS, verbose=True)
print(f"({time.perf_counter() - t0:.0f} s for the {loop['time'].size} laps)")
one = torch.as_tensor(TEST["rgb"][:1], device=DEVICE)
with torch.no_grad():
    t_net = middle(lambda: net(one))                             # one decision of the network
print(f"one decision: network {t_net * 1e3:.3f} ms on {DEVICE}, conventional rule {REF['t_rule_s'] * 1e3:.3f} ms on the CPU "
      f"(the network takes {t_net / REF['t_rule_s']:.1f} times as long)")
R = {"p_test": p_test, "w_test": w_test, "loop": loop, "t_net_s": t_net, "train_seconds": TRAIN_SECONDS,
     "n_weights": sum(p.numel() for p in net.parameters()), "device": DEVICE}   # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP11A_net.pt"))     # the trained weights, beside the report

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth (the labels `TEST` and the perfect-perception laps in `G`) and the reference model's results `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the eighteen questions.

In [ ]:
# What part E uses, all from the parts above
print("in R:", ", ".join(R))
print(f"perfect perception: mean lap {REF['perfect_time']:.2f} s; conventional rule {REF['t_rule_s'] * 1e3:.3f} ms per decision")

## E2 · TODO 4 · The goal metrics

Compute and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* on the held-out frames of levels 0 to 2, pooled: the share missed of each kind, the false stops, and the mean position error of each kind (where both say blocked) - for your network and for the conventional rule;
* the same at level 3, never trained on;
* in closed loop: the collisions at levels 0 to 2 (of 30 laps) and at level 3 (of 10); the mean lap time at levels 0 to 2 against perfect perception's, in per cent; the same for the conventional rule with the field at 1.5 m/s;
* the decision time of the network, the conventional rule's, and their ratio; the offline training time.

Then say in one line each whether the five requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>60}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP11A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the share missed per kind and level for your network and the conventional rule; the lap times per track at each level against perfect perception's; and the path of one lap on the track from above (`rt.draw_track(..., paths={...})`, the path from `rt.drive`).

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the eighteen questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 4 are about your network and its training, 5 to 9 about what it perceives, 10 to 14 about the car it drives, 15 to 17 ask you to use it, 18 concludes. The next cell writes `MP11A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its input (and any scaling or extra channel), its layers, channels and neurons, its outputs and the number of weights. Why this design for a 64 x 112 frame?": """
""",
    "2. Your training: the optimiser, learning rate and schedule, batch size, number of passes, any augmentation, and the offline time on the T4. How did the two loss terms evolve, and which one ended larger?": """
""",
    "3. Why is the position loss computed only on the blocked frames, and why is the position scaled by 0.5 m? What happened, or what would happen, if every frame were given a position of zero instead?": """
""",
    "4. The threshold on the probability of blocked: which did you use, and how do the share missed and the false stops at levels 0 - 2 change if you move it to 0.3 and to 0.7?": """
""",
    "5. The share missed of each kind at levels 0 - 2, for your network and the conventional rule. Which kind does the rule miss most, why, and what in the frame lets your network see it?": """
""",
    "6. False stops on free frames and on frames with only a shadow, for both. Why does the dark-pixel rule stop for shadows, and what does your network use to tell a shadow from a low dark block?": """
""",
    "7. The mean position error of each kind, for both. Which kind is placed worst, and where in the zone (near or far) is your position error largest? Explain with the row distances of section B2.": """
""",
    "8. Level 3, never trained on: your share missed, false stops and position errors there. What changed in the frames, and which of your network's answers broke first?": """
""",
    "9. Give one held-out frame your network gets wrong at level 2 (show it): what does it show, what did the network answer, and why might it have been fooled?": """
""",
    "10. Closed loop at levels 0 - 2: the collisions (of 30), the mean lap time, and the per cent against perfect perception's 18.13 s. Which tracks were slowest, and why?": """
""",
    "11. The conventional rule in closed loop: its collisions with the steer-around rule at 1.0 m/s and with the field at 1.5 m/s. Which kind of obstacle did it hit, and why does the faster planner not save it?": """
""",
    "12. Closed loop at level 3: how many collisions, and what caused them - missed obstacles, phantom obstacles or wrong positions? Use one lap's path or its sightings to show it.": """
""",
    "13. The smallest gap between the car and an obstacle over your 30 laps, and the planner's 10 cm margin: how much of the margin did your position errors use? What would you change if a gap were negative?": """
""",
    "14. The planner remembers an obstacle only once it is reported twice within a second, and ignores reports nearer than 0.7 m. Why both rules? Measure what the 10 laps at level 2 do without each (a planner class derived from rt.FieldAround with CONFIRM = 0, and one with NEAR = 0).": """
""",
    "15. Faster: drive the test tracks at levels 0 - 2 at 1.8 m/s, the R2's largest speed. The collisions, the mean lap and the per cent against perfect perception's at 1.8 m/s (rt.drive with rt.oracle gives it). Is it worth it?": """
""",
    "16. The decision time of your network on the T4 and on the CPU, the conventional rule's, and their ratio. The network is not faster than the rule: why is it still the better choice for this car, given the 100 ms between decisions?": """
""",
    "17. Train again with another seed (both the weights and the order of the frames). How much do the misses per kind, the position errors and the closed-loop results change? Is your result reproducible within the requirements?": """
""",
    "18. To conclude: on this car, with this camera, what limits the lap time - perception, the planner, or the bends where the camera sees little of the track ahead? What would you change first to go faster, and why?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP11A — Faster from A to B", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP11A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP11A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP11A_report.md into MP11A_report.pdf, with any figure
# saved as MP11A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP11A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP11A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP11A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP11A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass